In [13]:
from haystack_integrations.components.generators.ollama import OllamaChatGenerator

model_name = "gpt-oss:20b"
generator = OllamaChatGenerator(model=model_name,
                            url = "                            timeout = 30*60,
                            generation_kwargs={
                              "temperature": 0,
                              "num_ctx": 8192
                              })

In [15]:
from dataclasses import dataclass, field
from typing import Callable, Tuple

from haystack.dataclasses import ChatMessage
from haystack.tools import create_tool_from_function
from haystack.components.tools import ToolInvoker


@dataclass
class ToolCallingAgent:
    name: str = "ToolCallingAgent"
    llm: object = None
    instructions: str = "You are a helpful Agent"
    functions: list[Callable] = field(default_factory=list)

    def __post_init__(self):
        # system prompt
        self._system_message = ChatMessage.from_system(self.instructions)

        # convert python functions → tools
        self.tools = [
            create_tool_from_function(fun)
            for fun in self.functions
        ]

        # tool executor
        self._tool_invoker = ToolInvoker(
            tools=self.tools,
            raise_on_failure=False
        )

    def run(self, messages: list[ChatMessage]) -> list[ChatMessage]:

        # 1. LLM call
        agent_message = self.llm.run(
            messages=[self._system_message] + messages,
            tools=self.tools
        )["replies"][0]

        new_messages = [agent_message]

        # print response
        if agent_message.text:
            print(f"\n{self.name}: {agent_message.text}")

        # 2. no tool call → stop
        if not agent_message.tool_calls:
            return new_messages

        # 3. execute tools (IMPORTANT STEP 3)
        tool_results = self._tool_invoker.run(
            messages=[agent_message]
        )["tool_messages"]

        # 4. return tool outputs (STEP 4)
        new_messages.extend(tool_results)

        return new_messages


In [19]:
def add(a: int, b: int) -> int:
    return a + b

In [20]:
agent = ToolCallingAgent(
    llm = generator,
    functions=[add]   # pass your tool here
)

In [21]:
from haystack.dataclasses import ChatMessage

messages = [
    ChatMessage.from_user("What is 4 + 6? Use a tool.")
]

In [22]:
response = agent.run(messages)

In [ ]:
for msg in response:
    print(msg)

ChatMessage(_role=<ChatRole.ASSISTANT: 'assistant'>, _content=[ReasoningContent(reasoning_text='The user asks: "What is 4 + 6? Use a tool." We have a tool "functions.add" that takes a and b. We should call that.', extra={}), ToolCall(tool_name='add', arguments={'a': 4, 'b': 6}, id=None, extra=None)], _name=None, _meta={'model': 'gpt-oss:20b', 'done': True, 'total_duration': 4257555700, 'load_duration': 3094662000, 'prompt_eval_duration': 83325200, 'eval_duration': 953311900, 'logprobs': None, 'finish_reason': 'stop', 'completion_start_time': '2026-03-31T15:51:36.9529271Z', 'usage': {'completion_tokens': 63, 'prompt_tokens': 134, 'total_tokens': 197}})
ChatMessage(_role=<ChatRole.TOOL: 'tool'>, _content=[ToolCallResult(result='10', origin=ToolCall(tool_name='add', arguments={'a': 4, 'b': 6}, id=None, extra=None), error=False)], _name=None, _meta={})
